In [ ]:
import os
import numpy as np
import torch

# === 경로 설정 (필요에 따라 수정) ===
VERTS_PATH  = "D:/데이터 증강/features_verts/A0002_s80_features.npy"
LABEL_PATH  = "D:/데이터 증강/features_labels/A0002_s80_vertex_labels.npy"
SPIRAL_PATH = "spiral_9.npy"
WEIGHT_PATH = "dhkstjd.pth"  # best 모델 체크포인트
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === 1. 데이터 로드 ===
verts_np  = np.load(VERTS_PATH)
labels_np = np.load(LABEL_PATH)
verts     = torch.from_numpy(verts_np).float().unsqueeze(0).to(DEVICE)  # (1, V, C)

# === 2. spiral indices (학습 때와 동일하게) ===
spiral_all = np.load(SPIRAL_PATH, allow_pickle=True)
num_blocks = 4  # 학습 때와 꼭 맞춰야 함!
spiral_idx = [torch.from_numpy(spiral_all).long().to(DEVICE) for _ in range(num_blocks)]

feature_dim = verts_np.shape[1]

# === 3. 모델 정의 (학습 때와 구조 완전히 동일!) ===
# ImprovedSpiralNet, HybridSpiralNetPointNetTransformer, ... 정의부는 그대로 가져와야 함
class ImbalancedCTNoduleDataset(Dataset):
    def __init__(self, verts_dir, labels_dir, bases, augment=False):
        self.verts_dir = verts_dir
        self.labels_dir = labels_dir
        self.bases = bases
        self.augment = augment

        # 각 샘플의 positive 비율 계산 (가중치 샘플링용)
        self.sample_weights = []
        for base in bases:
            labels = np.load(os.path.join(labels_dir, base + '_vertex_labels.npy'))
            pos_ratio = labels.mean()
            # positive가 많은 샘플에 더 높은 가중치
            weight = min(pos_ratio * 10 + 0.1, 2.0)  # 최대 2배까지 가중치
            self.sample_weights.append(weight)

    def __len__(self):
        return len(self.bases)

    def get_sample_weights(self):
        return self.sample_weights

    def __getitem__(self, i):
        base = self.bases[i]
        verts = np.load(os.path.join(self.verts_dir, base + '_features.npy'))
        labels = np.load(os.path.join(self.labels_dir, base + '_vertex_labels.npy'))

        # 간단한 augmentation (결절이 있는 경우)
        if self.augment and labels.mean() > 0.01:
            # 작은 노이즈 추가
            noise = np.random.normal(0, 0.001, verts.shape)
            verts = verts + noise

        return torch.tensor(verts, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32), base

class SimplePointTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, ff_hidden=256, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_hidden),
            nn.ReLU(),
            nn.Linear(ff_hidden, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        # x: (B, V, C)
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out
        h2 = self.norm2(x)
        x = x + self.ff(h2)
        return x

class HybridSpiralNetPointNetTransformer(nn.Module):
    def __init__(self, spiralnet, in_channels, hidden_dim=256, out_dim=1, dropout=0.3,
                 transformer_heads=4, transformer_blocks=1):
        super().__init__()
        self.spiralnet = spiralnet  # 기존 SpiralNet 인스턴스

        # PointNet branch
        self.pointnet_mlp1 = nn.Linear(in_channels, hidden_dim)
        self.pointnet_bn1 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_mlp2 = nn.Linear(hidden_dim, hidden_dim)
        self.pointnet_bn2 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_dropout = nn.Dropout(dropout)

        # Transformer branch (Point Attention)
        self.transformer_blocks = nn.ModuleList([
            SimplePointTransformerBlock(hidden_dim, num_heads=transformer_heads, dropout=dropout)
            for _ in range(transformer_blocks)
        ])

        # 최종적으로 vertex별로 결합 (B, V, hidden*2)
        self.pointnet_head = nn.Linear(hidden_dim * 2, out_dim)

    def forward(self, x, spiral_idx):
        # x: (B, V, C)
        spiral_out = self.spiralnet(x, spiral_idx)  # (B, V)

        B, V, C = x.size()
        # ----- PointNet branch -----
        # Local MLP1
        feat1 = self.pointnet_mlp1(x)  # (B, V, hidden_dim)
        feat1 = feat1.view(B * V, -1)
        feat1 = self.pointnet_bn1(feat1)
        feat1 = F.relu(feat1)
        feat1 = feat1.view(B, V, -1)

        # Local MLP2
        feat2 = self.pointnet_mlp2(feat1)
        feat2 = feat2.view(B * V, -1)
        feat2 = self.pointnet_bn2(feat2)
        feat2 = F.relu(feat2)
        feat2 = feat2.view(B, V, -1)

        point_feat = self.pointnet_dropout(feat2)

        # --- Transformer Attention 추가 ---
        for block in self.transformer_blocks:
            point_feat = block(point_feat)  # (B, V, hidden_dim)

        # Global pooling
        global_feat, _ = torch.max(point_feat, dim=1, keepdim=True)  # (B, 1, hidden_dim)
        global_feat = global_feat.expand(-1, V, -1)  # (B, V, hidden_dim)

        # concat local+global features
        pn_feat = torch.cat([point_feat, global_feat], dim=2)  # (B, V, hidden_dim*2)
        pointnet_out = self.pointnet_head(pn_feat).squeeze(-1)  # (B, V)

        # Soft ensemble (SpiralNet + PointNet)
        out = (spiral_out + pointnet_out) / 2
        return out

# SpiralNet + PointNet 하이브리드 모델 클래스
class HybridSpiralNetPointNet(nn.Module):
    def __init__(self, spiralnet, in_channels, hidden_dim=256, out_dim=1, dropout=0.3):
        super().__init__()
        self.spiralnet = spiralnet  # 기존 SpiralNet 인스턴스

        # PointNet branch: (B, V, C) → (B, V, hidden) → (B, hidden)
        self.pointnet_mlp1 = nn.Linear(in_channels, hidden_dim)
        self.pointnet_bn1 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_mlp2 = nn.Linear(hidden_dim, hidden_dim)
        self.pointnet_bn2 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_dropout = nn.Dropout(dropout)
        # 최종적으로 vertex별로 결합 (B, V, hidden*2)
        self.pointnet_head = nn.Linear(hidden_dim * 2, out_dim)

    def forward(self, x, spiral_idx):
        # x: (B, V, C)
        spiral_out = self.spiralnet(x, spiral_idx)  # (B, V)

        B, V, C = x.size()
        # ----- PointNet branch -----
        # Local MLP1
        feat1 = self.pointnet_mlp1(x)  # (B, V, hidden_dim)
        feat1 = feat1.view(B * V, -1)  # (B*V, hidden_dim)
        feat1 = self.pointnet_bn1(feat1)
        feat1 = F.relu(feat1)
        feat1 = feat1.view(B, V, -1)  # (B, V, hidden_dim)

        # Local MLP2
        feat2 = self.pointnet_mlp2(feat1)
        feat2 = feat2.view(B * V, -1)
        feat2 = self.pointnet_bn2(feat2)
        feat2 = F.relu(feat2)
        feat2 = feat2.view(B, V, -1)  # (B, V, hidden_dim)

        point_feat = self.pointnet_dropout(feat2)  # (B, V, hidden_dim)

        # Global pooling
        global_feat, _ = torch.max(point_feat, dim=1, keepdim=True)  # (B, 1, hidden_dim)
        global_feat = global_feat.expand(-1, V, -1)  # (B, V, hidden_dim)

        # concat local+global features
        pn_feat = torch.cat([point_feat, global_feat], dim=2)  # (B, V, hidden_dim*2)
        pointnet_out = self.pointnet_head(pn_feat).squeeze(-1)  # (B, V)

        # Soft ensemble (SpiralNet + PointNet)
        out = (spiral_out + pointnet_out) / 2
        return out

class HybridSpiralNetMLP(nn.Module):
    def __init__(self, spiralnet, mlp_hidden=256, num_classes=1):
        super().__init__()
        self.spiralnet = spiralnet  # 기존 SpiralNet 구조
        in_dim = spiralnet.in_channels
        # MLP branch (예: 3-layer)
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, mlp_hidden//2),
            nn.ReLU(),
            nn.Linear(mlp_hidden//2, num_classes)
        )
        # 최종 head (concat된 feature로)
        self.head = nn.Sequential(
            nn.Linear(2 * num_classes, num_classes)  # spiral + mlp output concat
        )
    def forward(self, x, spiral_idx):
        # x: (B, V, C)
        # SpiralNet output (B, V)
        out_spiral = self.spiralnet(x, spiral_idx)
        # MLP output (B, V)
        out_mlp = self.mlp(x)
        # concat outputs
        out = torch.cat([out_spiral.unsqueeze(-1), out_mlp], dim=-1)
        out = self.head(out).squeeze(-1)
        return out

# ────────────────────────────────────────────────────
# SpiralNet Model (기존과 동일)
# ────────────────────────────────────────────────────
class SpiralConv(nn.Module):
    def __init__(self, in_channels, out_channels, spiral_len: int):
        super().__init__()
        self.spiral_len = spiral_len
        self.conv = nn.Conv1d(in_channels * spiral_len, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, spiral_idx):
        B, C, V = x.size()
        L = spiral_idx.size(1)
        idx = spiral_idx.unsqueeze(0).unsqueeze(0).expand(B, C, V, L)
        x_expand = x.unsqueeze(-1).expand(B, C, V, L)
        neigh = torch.gather(x_expand, 2, idx)
        neigh = neigh.reshape(B, C * L, V)
        out = self.conv(neigh)
        out = self.bn(out)
        return self.relu(out)

class SpiralBlock(nn.Module):
    def __init__(self, in_channels, out_channels, spiral_index, dropout=0.0):
        super().__init__()
        spiral_len = spiral_index.shape[1]
        self.spiral = SpiralConv(in_channels, out_channels, spiral_len)
        self.use_res = (in_channels == out_channels)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x, spiral_idx):
        out = self.spiral(x, spiral_idx)
        out = self.dropout(out)
        if self.use_res:
            return out + x
        else:
            return out

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.pool(x).squeeze(-1)
        w = self.fc(w).unsqueeze(-1)
        return x * w

class ImprovedSpiralNet(nn.Module):
    """개선된 SpiralNet with Multi-scale features"""
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_blocks, spiral_indices=None, dropout=0.0, use_se=True):
        super().__init__()
        self.spiral_indices = spiral_indices
        num_blocks = len(spiral_indices)
        self.use_se = use_se

        self.encoder = nn.ModuleList()
        for i in range(num_blocks):
            in_c = in_channels if i == 0 else hidden_channels
            out_c = hidden_channels
            spiral_index = spiral_indices[i]
            self.encoder.append(
                SpiralBlock(in_c, out_c, spiral_index, dropout=dropout)
            )

        if use_se:
            self.se_blocks = nn.ModuleList([
                SEBlock(hidden_channels) for _ in range(num_blocks)
            ])
        else:
            self.se_blocks = [nn.Identity()] * num_blocks

        self.skip_conns = nn.ModuleList()
        self.decoder = nn.ModuleList()
        for i in range(num_blocks - 1):
            self.skip_conns.append(nn.Conv1d(hidden_channels * 2, hidden_channels, kernel_size=1))
            spiral_index_dec = spiral_indices[num_blocks - 2 - i]
            self.decoder.append(
                SpiralBlock(hidden_channels, hidden_channels, spiral_index_dec, dropout=dropout)
            )

        # Multi-scale classifier
        self.classifier = nn.Sequential(
            nn.Conv1d(hidden_channels, hidden_channels // 2, kernel_size=1),
            nn.BatchNorm1d(hidden_channels // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_channels // 2, hidden_channels // 4, kernel_size=1),
            nn.BatchNorm1d(hidden_channels // 4),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_channels // 4, out_channels, kernel_size=1)
        )

    def forward(self, verts, indices):
        x = verts.permute(0, 2, 1)
        features = []

        for block, idx, se in zip(self.encoder, self.spiral_indices, self.se_blocks):
            x = block(x, idx)
            x = se(x)
            features.append(x)

        for skip_conv, dec_block, idx, feat in zip(
            self.skip_conns,
            self.decoder,
            reversed(self.spiral_indices[:-1]),
            reversed(features[:-1])
        ):
            x = torch.cat([x, feat], dim=1)
            x = skip_conv(x)
            x = dec_block(x, idx)

        out = self.classifier(x)
        return out.squeeze(1)

spiralnet = ImprovedSpiralNet(
    in_channels=feature_dim,
    hidden_channels=256,
    out_channels=1,
    num_blocks=num_blocks,
    spiral_indices=spiral_idx,
    dropout=0.3,
    use_se=True
).to(DEVICE)

model = HybridSpiralNetPointNetTransformer(
    spiralnet=spiralnet,
    in_channels=feature_dim,
    hidden_dim=256,
    out_dim=1,
    dropout=0.3,
    transformer_heads=4,
    transformer_blocks=1
).to(DEVICE)

# === 4. 가중치 로드 (best threshold 포함) ===
ckpt = torch.load(WEIGHT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
best_threshold = ckpt.get('threshold', 0.5)
print(f"불러온 모델의 best threshold: {best_threshold}")

model.eval()
with torch.no_grad():
    logits = model(verts, spiral_idx)         # (1, V)
    probs  = torch.sigmoid(logits).cpu().numpy().squeeze()  # (V,)
    preds  = (probs > best_threshold).astype(np.int64)

# === 5. 분포 확인 ===
print("예측 값 분포:", np.unique(preds, return_counts=True))
print("정답 값 분포:", np.unique(labels_np.squeeze(), return_counts=True))

# === 6. npy로 저장 ===
np.save(VERTS_PATH + f"_pred_thr{best_threshold:.2f}.npy", preds)
np.save(VERTS_PATH + "_probs.npy", probs)

import numpy as np
import matplotlib.pyplot as plt

# === 파일 경로 및 데이터 로드 ===
verts_np = np.load(VERTS_PATH)
labels_np = np.load(LABEL_PATH).squeeze()
preds = np.load(VERTS_PATH + f"_pred_thr{best_threshold:.2f}.npy")  # 위에서 저장한 파일
probs = np.load(VERTS_PATH + "_probs.npy")  # 확률값 (0~1)

is_pred = (preds == 1)
is_gt = (labels_np == 1)
is_both = is_pred & is_gt
is_pred_only = is_pred & (~is_gt)
is_gt_only = is_gt & (~is_pred)
is_none = (~is_pred) & (~is_gt)

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(verts_np[is_none,0], verts_np[is_none,1], verts_np[is_none,2],
           c='gray', s=3, alpha=0.1, label='Lung BG', zorder=1)
ax.scatter(verts_np[is_gt_only,0], verts_np[is_gt_only,1], verts_np[is_gt_only,2],
           c='yellow', s=18, alpha=1.0, label='GT Nodule', zorder=2)
ax.scatter(verts_np[is_pred_only,0], verts_np[is_pred_only,1], verts_np[is_pred_only,2],
           c='red', s=15, alpha=0.7, label='Predicted Nodule', zorder=3)
ax.scatter(verts_np[is_both,0], verts_np[is_both,1], verts_np[is_both,2],
           c='orange', s=25, alpha=1.0, label='Correct (Pred & GT)', zorder=4)

ax.set_title(f"Vertex-wise Prediction vs GT\n(threshold={best_threshold:.2f})", pad=20)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# === (1) npy 데이터 로드 ===
verts_np   = np.load(VERTS_PATH)                    # (V, 3) or (V, feature)
labels_np  = np.load(LABEL_PATH).squeeze()          # (V,)
preds      = np.load('C:/Users/<PC_A>/Desktop/spiral_torch/A0002_s80_features.npy_pred_thr0.70.npy').squeeze()           # (V,)

lung_pts   = verts_np  # 전체 포인트 (x, y, z)
true_nodule_pts = verts_np[labels_np == 1]
pred_nodule_pts = verts_np[preds == 1]

# === (2) 3D 시각화 ===
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# 전체 포인트 (연하게)
ax.scatter(lung_pts[:, 0], lung_pts[:, 1], lung_pts[:, 2],
           c='blue', s=1, alpha=0.3, label='Lung')

# 실제 결절 (노랑)
if len(true_nodule_pts) > 0:
    ax.scatter(true_nodule_pts[:, 0], true_nodule_pts[:, 1], true_nodule_pts[:, 2],
               c='yellow', s=20, edgecolors='none', label='True Nodule')

# 예측 결절 (빨강, 크고 진하게)
if len(pred_nodule_pts) > 0:
    ax.scatter(pred_nodule_pts[:, 0], pred_nodule_pts[:, 1], pred_nodule_pts[:, 2],
               c='red', s=10, alpha=1.0, edgecolors='none', label='Predicted Nodule')


ax.set_title('3D Visualization - Vertex Nodule Segmentation', fontsize=16)
ax.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()

